# Window functions: keep the rows, add the context

Practice notebook for the datafromzero SQL lesson **Window functions: keep the rows, add the context**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. GROUP BY collapses the rows

In [ ]:
con.sql("""
SELECT store_id, COUNT(*) AS orders
FROM orders
WHERE ordered_at >= '2026-01-01' AND ordered_at < '2026-01-08'
GROUP BY store_id
ORDER BY store_id
""").show()

### Step 2. A window function keeps them

In [ ]:
con.sql("""
SELECT order_id, store_id,
       COUNT(*) OVER (PARTITION BY store_id) AS store_orders
FROM orders
WHERE ordered_at >= '2026-01-01' AND ordered_at < '2026-01-08'
ORDER BY order_id
LIMIT 5
""").show()

### Step 3. Build the daily table (the first stage of every query below)

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, store_id, COUNT(*) AS orders
  FROM orders
  WHERE ordered_at >= '2026-01-01' AND ordered_at < '2026-01-08'
  GROUP BY day, store_id
)
SELECT day, store_id, orders
FROM daily
ORDER BY day, store_id
""").show()

### Step 4. Each cafe's share of the day

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, store_id, COUNT(*) AS orders
  FROM orders
  WHERE ordered_at >= '2026-01-01' AND ordered_at < '2026-01-08'
  GROUP BY day, store_id
)
SELECT day, store_id, orders,
       SUM(orders) OVER (PARTITION BY day) AS day_total,
       ROUND(orders * 100.0 / SUM(orders) OVER (PARTITION BY day), 1) AS share_pct
FROM daily
ORDER BY day, store_id
""").show()

### Step 5. A running total for each cafe

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, store_id, COUNT(*) AS orders
  FROM orders
  WHERE ordered_at >= '2026-01-01' AND ordered_at < '2026-01-08'
  GROUP BY day, store_id
)
SELECT store_id, day, orders,
       SUM(orders) OVER (PARTITION BY store_id ORDER BY day) AS running_total
FROM daily
ORDER BY store_id, day
""").show()

### Step 6. The default frame: predict first, then run

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, store_id, COUNT(*) AS orders
  FROM orders
  WHERE ordered_at >= '2026-01-01' AND ordered_at < '2026-01-08'
  GROUP BY day, store_id
)
SELECT day, store_id, orders,
       SUM(orders) OVER (ORDER BY day) AS default_frame,
       SUM(orders) OVER (ORDER BY day, store_id
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS row_by_row
FROM daily
ORDER BY day, store_id
""").show()

## Exercise 1. Each day's share of the cafe's week

The `daily` stage and the first lines are given. Add a column with each day's orders as a percent of that cafe's
orders for the week, rounded to 1 decimal.

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, store_id, COUNT(*) AS orders
  FROM orders
  WHERE ordered_at >= '2026-01-01' AND ordered_at < '2026-01-08'
  GROUP BY day, store_id
)
SELECT day, store_id, orders
FROM daily
ORDER BY store_id, day
""").show()

## Exercise 2. A running total for each payment method

Before you run anything, write one sentence in the first cell: how many rows do you expect, and why might it be
fewer than 21 (7 days times 3 payment methods)? Then build a `daily` stage by day and payment method over the same
week, and add a running total for each method.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague wants each cafe's share of each day's orders. The query runs and gives numbers that look reasonable.
Check that the shares of one day add to 100, then write the fix.

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS day, store_id, COUNT(*) AS orders
  FROM orders
  WHERE ordered_at >= '2026-01-01' AND ordered_at < '2026-01-08'
  GROUP BY day, store_id
)
SELECT day, store_id, orders,
       ROUND(orders * 100.0 / SUM(orders) OVER (), 1) AS share_pct
FROM daily
ORDER BY day, store_id
""").show()

In [ ]:
# Your corrected query: